In [5]:
from flask import Flask, request, jsonify
from flask_restx import Api, Resource, reqparse
import joblib
import numpy as np
import requests
import os

# ------------------------------------------------------------
# Configuración (cambia las URLs por las reales de tus archivos en GitHub)
# ------------------------------------------------------------
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/CrisSCon/MIAD_ML_NLP_2026_Disponibilizacion1/main"

MODEL_URL = f"{GITHUB_RAW_BASE}/model.pkl"
FEATURES_URL = f"{GITHUB_RAW_BASE}/features.pkl"
ENCODINGS_URL = f"{GITHUB_RAW_BASE}/target_encodings.pkl"

# ------------------------------------------------------------
# Descargar y cargar los artefactos desde GitHub (o usar caché local)
# ------------------------------------------------------------
def download_file(url, local_path):
    """Descarga un archivo si no existe localmente."""
    if not os.path.exists(local_path):
        print(f"Descargando {local_path} ...")
        r = requests.get(url)
        r.raise_for_status()
        with open(local_path, "wb") as f:
            f.write(r.content)
    else:
        print(f"Usando archivo local: {local_path}")

download_file(MODEL_URL, "model.pkl")
download_file(FEATURES_URL, "features.pkl")
download_file(ENCODINGS_URL, "target_encodings.pkl")

model = joblib.load("model.pkl")
FEATURE_NAMES = joblib.load("features.pkl")
target_encodings = joblib.load("target_encodings.pkl")
global_mean = target_encodings["global_mean"]

# ------------------------------------------------------------
# Funciones auxiliares (reproducen el feature engineering)
# ------------------------------------------------------------
def create_raw_features(row):
    """Aplica las transformaciones del dataset original."""
    d = row.copy()
    d['durlog'] = np.log1p(d['duration_ms'])
    d['eneloud'] = d['energy'] * d['loudness']
    d['danval'] = d['danceability'] * d['valence']
    d['isinstr'] = int(d['instrumentalness'] > 0.45)
    d['expl'] = int(d['explicit'])
    return d

def add_target_encodings(row):
    """Agrega las columnas de target encoding."""
    artist = row.get('artists', '')
    genre = row.get('track_genre', '')
    album = row.get('album_name', '')
    
    row['artistste'] = target_encodings['artists'].get(artist, global_mean)
    row['track_genrete'] = target_encodings['track_genre'].get(genre, global_mean)
    row['album_namete'] = target_encodings['album_name'].get(album, global_mean)
    return row

# ------------------------------------------------------------
# Flask y Swagger
# ------------------------------------------------------------
app = Flask(__name__)
api = Api(app, version='1.0', title='Song Popularity API')
ns = api.namespace('predict', description='Predicción de popularidad')

# Parser para GET (todos los parámetros en la URL)
parser = reqparse.RequestParser()
numeric_params = [
    'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness',
    'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness',
    'valence', 'tempo'
]
for param in numeric_params:
    parser.add_argument(param, type=float, required=True, location='args')
# Parámetros categóricos (para target encoding)
parser.add_argument('artists', type=str, required=True, location='args')
parser.add_argument('track_genre', type=str, required=True, location='args')
parser.add_argument('album_name', type=str, required=True, location='args')

@ns.route('/')
class PredictPopularity(Resource):
    @ns.expect(parser, validate=True)
    def get(self):
        """Predice la popularidad usando GET con parámetros en la URL."""
        args = parser.parse_args()
        return self._predict(args)

    def post(self):
        """Predice la popularidad usando POST con JSON."""
        data = request.get_json()
        return self._predict(data)

    def _predict(self, args):
        # 1. Convertir a diccionario
        row = {k: v for k, v in args.items()}
        # 2. Crear features derivados
        row = create_raw_features(row)
        # 3. Añadir target encodings
        row = add_target_encodings(row)
        # 4. Construir array en el orden exacto
        X = np.array([row[name] for name in FEATURE_NAMES]).reshape(1, -1)
        # 5. Predecir y clip
        pred = model.predict(X)[0]
        pred = float(np.clip(pred, 0, 100))
        return {"popularity": pred}, 200

if __name__ == '__main__':
    app.run(debug=True, use_reloader=False, host='0.0.0.0', port=5000)

Descargando model.pkl ...
Descargando features.pkl ...
Descargando target_encodings.pkl ...
 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.0.13:5000
Press CTRL+C to quit
C:\Users\delfi\anaconda3\envs\notebook\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\delfi\anaconda3\envs\notebook\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
127.0.0.1 - - [25/Apr/2026 14:33:17] "GET /predict/?duration_ms=211533&explicit=0&danceability=0.305&energy=0.849&key=9&loudness=-10.796&mode=1&speechiness=0.0393&acousticness=0.000282&instrumentalness=0.000784&liveness=0.129&valence=0.32&tempo=65.952&artists=Love+and+Rockets&track_genre=goth&album_name=Love+and+Rockets HTTP/1.1" 200 -
127.0.0.1 - - [25/Apr/2026 14:33:18] "GET /predict/?duration_ms=211533&explicit=0&danceability=0.305&energy=0.849&ke